In [10]:
import torch
import pandas as pd
from datacleaner import build_test_dataloader
from model import SimpleCNN

In [11]:
# 1. Load the Test Data using our updated utility
test_loader, sub_df = build_test_dataloader(
    sample_sub_path='./sample_submission.csv', 
    img_dir='./raw_data/test/test', 
    batch_size=32
)

Test Pipeline Ready! Images to predict: 29443


### Loading the Model

In [12]:
# 2. Setup the Device and Load the Trained Model

# Checking of GPU is available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# number of classes
model = SimpleCNN(num_classes=8)

# Loading the Model
model.load_state_dict(torch.load('./outputs/baseline_model_v1.pth', map_location=device))
model.to(device)
model.eval() # CRITICAL: Turns off training features like dropout

SimpleCNN(
  (conv1): Conv2d(3, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool1): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv2): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (fc1): Linear(in_features=50176, out_features=512, bias=True)
  (fc2): Linear(in_features=512, out_features=8, bias=True)
  (dropout): Dropout(p=0.5, inplace=False)
)

### Defining our Class back to string to put in csv

In [13]:
# 3. Define our mapping dictionary (Numbers back to Strings)
idx_to_class = {
    0: 'CatEye', 1: 'CatFace', 2: 'CatMouth', 3: 'CatNose',
    4: 'DogEye', 5: 'DogFace', 6: 'DogMouth', 7: 'DogNose'
}

### Running the Model

In [14]:
print("Generating predictions...")
predictions = []

with torch.no_grad(): 
    for images, ids in test_loader:
        images = images.to(device)
        outputs = model(images)
        
        # Find the index of the highest score
        _, predicted_idx = torch.max(outputs, 1)
        
        # Only save the string predictions
        for idx in predicted_idx.cpu().numpy():
            predictions.append(idx_to_class[idx])

Generating predictions...


### Saving the Results

In [15]:
sub_df['class'] = predictions

submission_path = './outputs/my_first_submission.csv'
sub_df.to_csv(submission_path, index=False)

print(f"Success! Submission fixed and saved to {submission_path}")
print(sub_df.head())

Success! Submission fixed and saved to ./outputs/my_first_submission.csv
   image_id     class
0         1    DogEye
1         2  DogMouth
2         3   DogNose
3         4   DogFace
4         5  CatMouth
